# Caso 3: AVANZADO - Calidad de Datos SISMEPRE (Impuesto Predial Municipalidades)

---

## Contexto del Proyecto

### Descripción del Problema
El **Ministerio de Economía y Finanzas (MEF)**, a través del sistema **SISMEPRE**,
registra el presupuesto y ejecución del impuesto predial de las municipalidades
peruanas. Al consolidar los datos, se detectan inconsistencias en los registros
de respuestas, valores faltantes en campos clave, y formatos irregulares que
impiden analizar correctamente el cumplimiento de metas de recaudación predial
por municipalidad.

### Objetivo Analítico
Implementar un pipeline robusto de calidad sobre los datos del SISMEPRE para:
- Detectar y corregir automáticamente valores faltantes e inconsistentes
- Registrar cada problema en un log de auditoría documentado
- Generar métricas de calidad cuantificables antes y después de la limpieza
- Producir el archivo Silver reproducible y escalable para el JOIN con SIAF y RENAMU

### Impacto de la Mala Calidad de Datos
- **Financiero**: Registros de recaudación predial incorrectos distorsionan
el análisis de cumplimiento de metas por municipalidad
- **Operativo**: Duplicados o respuestas inválidas generan conteos incorrectos
de municipalidades que reportaron su impuesto predial
- **Estratégico**: Decisiones de política tributaria municipal basadas en datos
incompletos pueden beneficiar o perjudicar municipios incorrectamente

---

## Dimensiones de Calidad a Corregir

En este caso aplicaremos correcciones avanzadas sobre:

1. **Completitud**: Imputación con mediana por PREGUNTA_ID con logging
2. **Exactitud**: Corrección de valores negativos con registro en auditoría
3. **Consistencia**: Validación automática entre SEC_EJEC y SEC_EJEC_1
4. **Integridad**: Validación de ANO_APLICACION y SEC_EJEC contra rangos válidos
5. **Razonabilidad**: Detección de anomalías con IQR por PREGUNTA_ID
6. **Oportunidad**: Validación automática de cobertura temporal
7. **Unicidad**: Deduplicación por clave de negocio con logging
8. **Validez**: Pipeline de validación automática de ESTADO y CLASIFICACION

---

In [1]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


In [2]:
# Cargar datos SISMEPRE
df_avanzado = pd.read_csv('SISMEPRE_CAPA_SILVER.csv', encoding='latin-1')

print(f"✅ Dataset SISMEPRE cargado correctamente")

✅ Dataset SISMEPRE cargado correctamente


---

# SOLUCIÓN NIVEL AVANZADO - SISMEPRE

## Objetivo
Implementar un pipeline robusto y automatizado de calidad de datos:
- Validaciones basadas en reglas del sistema SISMEPRE
- Detección automática de anomalías en respuestas
- Logging de problemas para auditoría
- Métricas de calidad cuantificables
- Proceso reproducible y escalable

In [3]:
quality_log = []

print(f"Registros iniciales: {len(df_avanzado):,}\n")

Registros iniciales: 100,000



In [4]:
class DataQualityValidator:

    def __init__(self, df):
        self.df = df.copy()
        self.initial_count = len(df)
        self.now = datetime.now()
        self.quality_log = []

    # LOGGING PROFESIONAL
    def log_issue(self, dimension, rule, affected_rows, action):
        self.quality_log.append({
            "timestamp":     self.now,
            "dimension":     dimension,
            "rule":          rule,
            "affected_rows": int(affected_rows),
            "action":        action
        })

    # 1. COMPLETITUD
    def validate_completeness(self):
        criticas = ['SEC_EJEC', 'ANO_APLICACION', 'PREGUNTA_ID']
        for col in criticas:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                self.df = self.df.loc[~mask]
                self.log_issue("completitud", f"{col}_null", count, "drop")

        # Imputación por mediana por PREGUNTA_ID
        for col in ['RESPUESTA_DECIMAL', 'RESPUESTA_ENTERO']:
            mask  = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                fill = self.df.groupby('PREGUNTA_ID')[col].transform('median')
                self.df[col] = self.df[col].fillna(fill).fillna(0)
                self.log_issue("completitud", f"{col}_null", count,
                               "fill_median_pregunta_id")

        # RESPUESTA_TEXTO nulo: imputar con 'SIN RESPUESTA'
        mask  = self.df['RESPUESTA_TEXTO'].isna()
        count = mask.sum()
        if count > 0:
            self.df['RESPUESTA_TEXTO'] = self.df['RESPUESTA_TEXTO'].fillna('SIN RESPUESTA')
            self.log_issue("completitud", "RESPUESTA_TEXTO_null", count, "fill_sin_respuesta")

    # 2. EXACTITUD
    def validate_accuracy(self):
        # RESPUESTA_DECIMAL negativa
        mask  = self.df['RESPUESTA_DECIMAL'] < 0
        count = mask.sum()
        if count > 0:
            median = self.df.loc[~mask, 'RESPUESTA_DECIMAL'].median()
            self.df.loc[mask, 'RESPUESTA_DECIMAL'] = median
            self.log_issue("exactitud", "RESPUESTA_DECIMAL_negativa",
                           count, f"median:{median}")

        # RESPUESTA_ENTERO negativa
        mask  = self.df['RESPUESTA_ENTERO'] < 0
        count = mask.sum()
        if count > 0:
            median = self.df.loc[~mask, 'RESPUESTA_ENTERO'].median()
            self.df.loc[mask, 'RESPUESTA_ENTERO'] = median
            self.log_issue("exactitud", "RESPUESTA_ENTERO_negativa",
                           count, f"median:{median}")

        # PREGUNTA_ID <= 0
        mask  = self.df['PREGUNTA_ID'] <= 0
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("exactitud", "PREGUNTA_ID_invalida", count, "drop")

    # 3. CONSISTENCIA
    def validate_consistency(self):
        # SEC_EJEC_1 debe coincidir con SEC_EJEC
        mask  = self.df['SEC_EJEC'] != self.df['SEC_EJEC_1']
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'SEC_EJEC_1'] = self.df.loc[mask, 'SEC_EJEC']
            self.log_issue("consistencia", "SEC_EJEC_vs_SEC_EJEC_1",
                           count, "corrected")

        # FORMULARIO_ID_1 debe coincidir con FORMULARIO_ID
        mask  = self.df['FORMULARIO_ID'] != self.df['FORMULARIO_ID_1']
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'FORMULARIO_ID_1'] = self.df.loc[mask, 'FORMULARIO_ID']
            self.log_issue("consistencia", "FORMULARIO_ID_vs_FORMULARIO_ID_1",
                           count, "corrected")

        # Estandarizar ESTADO y CLASIFICACION
        self.df['ESTADO']        = self.df['ESTADO'].str.upper().str.strip()
        self.df['CLASIFICACION'] = self.df['CLASIFICACION'].str.upper().str.strip()

    # 4. INTEGRIDAD
    def validate_integrity(self):
        # SEC_EJEC <= 0
        mask  = self.df['SEC_EJEC'] <= 0
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "SEC_EJEC_invalido", count, "drop")

        # ANO_APLICACION fuera de rango
        mask  = (self.df['ANO_APLICACION'] < 2010) | \
                (self.df['ANO_APLICACION'] > 2026)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "ANO_APLICACION_fuera_rango",
                           count, "drop")

    # 5. RAZONABILIDAD
    def validate_reasonableness(self):
        # IQR sobre RESPUESTA_DECIMAL
        Q1  = self.df['RESPUESTA_DECIMAL'].quantile(0.25)
        Q3  = self.df['RESPUESTA_DECIMAL'].quantile(0.75)
        IQR = Q3 - Q1
        upper = Q3 + 3 * IQR

        mask  = self.df['RESPUESTA_DECIMAL'] > upper
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("razonabilidad", "RESPUESTA_DECIMAL_outlier",
                           count, f"drop_limite_{upper:.2f}")

        # RESPUESTA_ENTERO outlier
        Q1  = self.df['RESPUESTA_ENTERO'].quantile(0.25)
        Q3  = self.df['RESPUESTA_ENTERO'].quantile(0.75)
        IQR = Q3 - Q1
        upper = Q3 + 3 * IQR

        mask  = self.df['RESPUESTA_ENTERO'] > upper
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("razonabilidad", "RESPUESTA_ENTERO_outlier",
                           count, f"drop_limite_{upper:.2f}")

    # 6. OPORTUNIDAD
    def validate_timeliness(self):
        ano_actual = datetime.now().year
        mask  = self.df['ANO_APLICACION'] > ano_actual
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("oportunidad", "ANO_APLICACION_futuro", count, "drop")

        mask  = self.df['ANO_APLICACION'] < 2010
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("oportunidad", "ANO_APLICACION_antiguo", count, "drop")

    # 7. UNICIDAD
    def validate_uniqueness(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        count   = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_exactos", count, "drop")

        cols_clave = ['SEC_EJEC', 'ANO_APLICACION', 'PREGUNTA_ID']
        before     = len(self.df)
        self.df    = self.df.drop_duplicates(subset=cols_clave, keep='first')
        count      = before - len(self.df)
        if count > 0:
            self.log_issue("unicidad", "duplicados_clave_negocio", count, "drop")

    # 8. VALIDEZ
    def validate_validity(self):
        # ESTADO: solo A o I
        estados_validos = ['A', 'I']
        mask  = ~self.df['ESTADO'].isin(estados_validos)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("validez", "ESTADO_invalido", count, "drop")

        # CLASIFICACION: solo A o B
        clasif_validas = ['A', 'B']
        mask  = ~self.df['CLASIFICACION'].isin(clasif_validas)
        count = mask.sum()
        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("validez", "CLASIFICACION_invalida", count, "drop")

    # PIPELINE
    def run(self):
        print("\n🔄 Ejecutando pipeline de calidad SISMEPRE...")
        self.validate_completeness()
        print("   ✅ Completitud")
        self.validate_accuracy()
        print("   ✅ Exactitud")
        self.validate_consistency()
        print("   ✅ Consistencia")
        self.validate_integrity()
        print("   ✅ Integridad")
        self.validate_reasonableness()
        print("   ✅ Razonabilidad")
        self.validate_timeliness()
        print("   ✅ Oportunidad")
        self.validate_uniqueness()
        print("   ✅ Unicidad")
        self.validate_validity()
        print("   ✅ Validez")
        print("\n✅ Pipeline completado")
        return self.df

    # MÉTRICAS
    def report(self):
        final_count = len(self.df)
        perdida     = self.initial_count - final_count
        pct_perdida = (perdida / self.initial_count * 100) if self.initial_count > 0 else 0

        print("\n" + "=" * 60)
        print("   REPORTE FINAL - PIPELINE AVANZADO SISMEPRE")
        print("=" * 60)
        print(f"\n📊 Registros iniciales : {self.initial_count:,}")
        print(f"📊 Registros finales   : {final_count:,}")
        print(f"📊 Registros perdidos  : {perdida:,} ({pct_perdida:.2f}%)")

        log_df = pd.DataFrame(self.quality_log)
        if len(log_df) > 0:
            print("\n📋 Resumen por dimensión:")
            print("-" * 60)
            print(log_df.groupby("dimension")["affected_rows"].sum().to_string())
            print("-" * 60)
            print(f"\n📋 Log completo de auditoría:")
            print(log_df[['dimension', 'rule',
                           'affected_rows', 'action']].to_string(index=False))
        else:
            print("\n✅ No se registraron problemas en el log")

        return log_df

In [5]:
# Ejecutar pipeline avanzado de calidad SISMEPRE
validator = DataQualityValidator(df_avanzado)
df_clean  = validator.run()
log_df    = validator.report()


🔄 Ejecutando pipeline de calidad SISMEPRE...
   ✅ Completitud
   ✅ Exactitud
   ✅ Consistencia
   ✅ Integridad
   ✅ Razonabilidad
   ✅ Oportunidad
   ✅ Unicidad
   ✅ Validez

✅ Pipeline completado

   REPORTE FINAL - PIPELINE AVANZADO SISMEPRE

📊 Registros iniciales : 100,000
📊 Registros finales   : 13,024
📊 Registros perdidos  : 86,976 (86.98%)

📋 Resumen por dimensión:
------------------------------------------------------------
dimension
consistencia     199915
razonabilidad      3400
unicidad          65150
validez           18426
------------------------------------------------------------

📋 Log completo de auditoría:
    dimension                             rule  affected_rows           action
 consistencia           SEC_EJEC_vs_SEC_EJEC_1          99915        corrected
 consistencia FORMULARIO_ID_vs_FORMULARIO_ID_1         100000        corrected
razonabilidad        RESPUESTA_DECIMAL_outlier            425 drop_limite_0.00
razonabilidad         RESPUESTA_ENTERO_outlier     

In [6]:
# Generar reporte final
log_df

,timestamp,dimension,rule,affected_rows,action
0,2026-05-21 20:55:05.483211,consistencia,SEC_EJEC_vs_SEC_EJEC_1,99915,corrected
1,2026-05-21 20:55:05.483211,consistencia,FORMULARIO_ID_vs_FORMULARIO_ID_1,100000,corrected
2,2026-05-21 20:55:05.483211,razonabilidad,RESPUESTA_DECIMAL_outlier,425,drop_limite_0.00
3,2026-05-21 20:55:05.483211,razonabilidad,RESPUESTA_ENTERO_outlier,2975,drop_limite_0.00
4,2026-05-21 20:55:05.483211,unicidad,duplicados_exactos,55375,drop
5,2026-05-21 20:55:05.483211,unicidad,duplicados_clave_negocio,9775,drop
6,2026-05-21 20:55:05.483211,validez,CLASIFICACION_invalida,18426,drop


In [7]:
df_clean.to_parquet("sismepre_silver_avanzado.parquet", index=False)
print(f"✅ Archivo guardado: sismepre_silver_avanzado.parquet")

✅ Archivo guardado: sismepre_silver_avanzado.parquet


### Conclusiones de la Solución Avanzada - SISMEPRE

**Ventajas:**
- Pipeline automatizado y reproducible aplicable también a SIAF y RENAMU
- Logging completo para auditoría: cada problema queda registrado con
  dimensión, regla, cantidad de registros afectados y acción tomada
- Trata las 8 dimensiones de calidad sobre datos reales del SISMEPRE
- Trazabilidad completa de cada decisión de limpieza

**Características clave:**
- Modular: cada dimensión es un método independiente que puede
  modificarse sin afectar el resto del pipeline
- Métricas cuantificables: el reporte final muestra pérdida exacta
  de registros por dimensión
- Reglas de negocio explícitas del sistema SISMEPRE (ESTADO A/I,
  CLASIFICACION A/B, rango ANO_APLICACION 2010-2026)
- Escalable: la clase DataQualityValidator puede instanciarse
  con el CSV de RENAMU sin cambiar la estructura